In [1]:
import pandas as pd
import numpy as np

compound_to_module = {
    "AT9283":"JAK", "Apicidin":"HDAC", "BI 2536":"PLK", "BMS-536924":"IGF1R",
    "BRD-K76674262":"STAT3", "Bisindolylmaleimide II":"PKC", "Brefeldin A":"ATPase",
    "CGP-60474":"CDK", "CYT-387":"JAK", "Calcitriol":"VDR", "Crizotinib":"ALK",
    "Decitabine":"DNMT", "Deforolimus":"MTOR", "Dexamethasone":"GR",
    "Dorsomorphin":"AMPK", "EX-527":"SIRT", "Entospletinib":"SYK", "Etomoxir":"CPT1",
    "Forskolin":"ADCY", "Fostamatinib":"SYK", "Geldanamycin":"HSP90", "I-BET 762":"BRD",
    "JNK-9L (JNK Inhibitor)":"JNK", "NFkB Activation Inhibitor II":"NFkB",
    "NVP-AUY922":"HSP90", "NVP-BHG 712":"EPH", "Neratinib":"ERBB", "PHA-793887":"CDK",
    "Penfluridol":"DRD", "Pevonedistat":"NAE", "Pomalidomide":"CRBN", "Ponatinib":"ABL",
    "Purmorphamine":"SMO", "QS-11":"ARFGAP", "Rebastinib":"ABL", "Resveratrol":"LCK",
    "Rucaparib":"PARP", "Sapanisertib":"MTOR", "Selumetinib":"MEK", "Serdemetan":"MDM2",
    "Sorafenib":"RAF", "TG-101348":"JAK", "TWS-119":"GSK3", "Temsirolimus":"MTOR",
    "Thapsigargin":"SERCA", "Tozasertib":"Aurora", "VX-11e":"ERK", "Veliparib":"PARP",
    "Withaferin A":"NFkB", "Wortmannin":"PI3K", "YM-155":"BIRC5", "belinostat":"HDAC",
    "camptothecin":"TOP1", "navitoclax":"BCL", "panobinostat":"HDAC", "ruxolitinib":"JAK",
}

In [ ]:
CD8_DIR = "/mnt/R0/Projects/POIAZ/snugent/Notebooks/output_folder/CD8_Created_Files"

dpd = pd.read_csv(f"{CD8_DIR}/dpd_sum_per_compound_raw.csv")
dpd_col = "dpd"
dpd["module"] = dpd["compound_name"].map(compound_to_module)
dpd_inh = dpd[dpd["mechanism"] == "Inhibitor"].dropna(subset=["module"]).copy()

# Top 4 + bottom 4 DRUGS → their modules
top4    = dpd_inh.nlargest(4,  dpd_col)
bottom4 = dpd_inh.nsmallest(4, dpd_col)
selected_modules = pd.concat([top4, bottom4])["module"].unique().tolist()
drug_set = dpd_inh[dpd_inh["module"].isin(selected_modules)].copy()
# Build drug-module-IC50 table
df = pd.read_csv(f"{CD8_DIR}/cd8_limma_merged_filtered_targets_ic50_dpd.csv")
def parse_ic50(val):
    if pd.isna(val) or str(val).strip().lower() == "unknown":
        return np.nan
    try:
        v = str(val).replace("/", ",")
        vals = [float(x) for x in v.split(",") if x.strip()]
        return round(np.exp(np.mean(np.log(vals))), 4)
    except Exception:
        return np.nan
rows = []
for _, r in drug_set.iterrows():
    compound = r["compound_name"]
    ic50_raw = df[df["compound_name"] == compound]["IC50_nM"].iloc[0] \
               if (df["compound_name"] == compound).any() else np.nan
    rows.append({"compound_name": compound, "module": r["module"],
                 "IC50_nM": parse_ic50(ic50_raw), "dpd": r[dpd_col],
                 "mechanism": r["mechanism"]})
drug_module_ic50 = pd.DataFrame(rows).sort_values("module")
drug_module_ic50.to_csv("top4_bottom4_selected_modules_drugs_ic50.csv", index=False)


In [ ]:
# Load the collapsed-module selection from Cell 2
IC50_df = pd.read_csv("top4_bottom4_selected_modules_drugs_ic50.csv")\
    .rename(columns={"module": "Module", "compound_name": "Drug", "IC50_nM": "IC50"})
IC50_df["IC50"] = IC50_df["IC50"] / 1000          # nM → uM
IC50_df = IC50_df.set_index("Module").drop_duplicates()
IC50_df_reset = IC50_df.reset_index()

modules = IC50_df.index.unique().tolist()
drugs   = IC50_df_reset["Drug"].drop_duplicates().tolist()

# Dose and mechanism per compound
dose_info = df.groupby("compound_name")["dose_uM"].first()
mech_info = df.groupby("compound_name")["mechanism"].first()

GAMMA = 1.0   # activation coefficient, applied only where mechanism == "Activator"

# Build matrices
inhib_conc_matrix = np.zeros((len(modules), len(drugs)))
ic50_matrix       = np.zeros((len(modules), len(drugs)))
gamma_matrix      = np.zeros((len(modules), len(drugs)))   # 0 = inhibitor, >0 = activator

for i, module in enumerate(modules):
    drugs_for_module = IC50_df_reset["Drug"][IC50_df_reset["Module"] == module].tolist()
    for drug in drugs_for_module:
        ic50 = IC50_df_reset["IC50"][IC50_df_reset["Drug"] == drug].values
        dose = dose_info.get(drug, np.nan)
        if ic50.size == 0 or pd.isna(dose) or drug not in drugs:
            continue
        j = drugs.index(drug)
        inhib_conc_matrix[i, j] = dose
        ic50_matrix[i, j]       = ic50[0]
        gamma_matrix[i, j]      = GAMMA if mech_info.get(drug, "Inhibitor") == "Activator" else 0.0

inhib_conc_df = pd.DataFrame(inhib_conc_matrix, index=modules, columns=drugs)
ic50_df_out   = pd.DataFrame(ic50_matrix,       index=modules, columns=drugs)
pert_df       = pd.DataFrame(np.where(inhib_conc_matrix != 0, 1, 0),
                             index=modules, columns=drugs)

g_matrix = np.ones((len(modules), len(drugs)))

for i in range(len(modules)):
    for j in range(len(drugs)):
        ic50 = ic50_matrix[i, j]
        dose = inhib_conc_matrix[i, j]

        if ic50 <= 0 or dose == 0:
            g_matrix[i, j] = 1.0          # no perturbation → g = 1
            continue

        dratio = dose / ic50

        if mechanism_matrix[i, j] == "Activator":
            g_matrix[i, j] = (1 + GAMMA * dratio) / (1 + dratio)   # activator: (1 + gamma*I/R) / (1 + k/IC50)
        else:
            g_matrix[i, j] = 1 / (1 + dratio)                  # inhibitor: 1/(1 + k/IC50)

g_df = pd.DataFrame(g_matrix, index=modules, columns=drugs)
inhib_conc_df.to_csv("inhib_conc_matrix_top4_bottom4.csv")   # or _min2.csv in that cell
ic50_df_out.to_csv("ic50_matrix_top4_bottom4.csv")            # or _min2.csv in that cell
pert_df.to_csv("pert_matrix_top4_bottom4.csv")                # or _min2.csv in that cell
g_df.to_csv("g_matrix_top4_bottom4.csv")                      # or _min2.csv in that cell
